# Online2 Solver Logs Analysis

Questo notebook visualizza i log prodotti dal scheduler (`online2`) a ogni esecuzione del solver.

Contiene:
- andamento del tempo medio di processing per richiesta e per batch
- un grafico stacked per ciascuna esecuzione del solver con assegnamenti, errori medi per slot e capacity levels

In [ ]:
%matplotlib inline
import pathlib
import sys

cwd = pathlib.Path.cwd().resolve()
candidates = [cwd, cwd.parent]
for candidate in candidates:
    if (candidate / 'config.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import pandas as pd

import config
import importlib
import visualize_solver_logs as viz
importlib.reload(viz)
from visualize_solver_logs import (
    load_solver_logs,
    load_infeasibility_debug_logs,
    plot_processing_times,
    plot_carbon_intensity_by_slot,
    plot_error_window_trend,
    plot_solver_execution_stacked,
    plot_infeasibility_overview,
    plot_infeasibility_event,
    select_run_ids,
)


In [ ]:
# Run selection options for stacked assignment plots
# all | first_per_slot | last_per_slot | all_for_slot
RUN_SELECTION_MODE = "first_per_slot"
TARGET_SLOT = None  # used only when RUN_SELECTION_MODE == "all_for_slot"

# Strict-infeasible debug event selection
# None => latest event
INFEASIBILITY_EVENT_ID = None

runs_df, assignments_df, slot_metrics_df = load_solver_logs(
    runs_file=config.SOLVER_RUNS_FILE,
    assignments_file=config.SOLVER_ASSIGNMENTS_FILE,
    slot_metrics_file=config.SOLVER_SLOT_METRICS_FILE,
    only_runs_with_assignments=True,
)
debug_df = load_infeasibility_debug_logs(config.SOLVER_INFEASIBLE_DEBUG_FILE)

selected_run_ids = select_run_ids(
    runs_df,
    mode=RUN_SELECTION_MODE,
    target_slot=TARGET_SLOT,
) if not runs_df.empty else []

print(f"runs: {len(runs_df)}")
print(f"assignments rows: {len(assignments_df)}")
print(f"slot metrics rows: {len(slot_metrics_df)}")
print(f"strict-infeasible debug events: {len(debug_df)}")
print(f"selected stacked plots: {len(selected_run_ids)}")

display(runs_df.head())
if not debug_df.empty:
    display(debug_df.tail(5))


## 1) Performance, carbon intensity e trend errore window

Grafici riassuntivi:
- tempo medio per richiesta e tempo totale per batch
- carbon intensity per slot
- andamento di `error_window_avg_after` (modello usato dal solver) vs threshold
- confronto con `error_window_avg_after_real` (solo richieste reali, senza prehistory virtuale)


In [ ]:
if runs_df.empty:
    print("Nessun dato disponibile in SOLVER_RUNS_FILE")
else:
    fig = plot_processing_times(runs_df)
    plt.show()

fig = plot_carbon_intensity_by_slot(slot_metrics_df)
plt.show()

if runs_df.empty:
    print("Nessun dato disponibile per il trend errore window")
else:
    fig = plot_error_window_trend(runs_df)
    plt.show()


## 2) Assegnamenti per esecuzione solver (stacked bars)

Ogni grafico mostra **tutte** le richieste attive in quel run (non solo le N nuove).
Le richieste nuove del run sono evidenziate, quelle preesistenti sono più trasparenti.

Nota: la linea `avg error per slot` è una metrica **per singolo slot** e può essere > threshold.
Il vincolo del solver è sulla media errore della **finestra [t-W, t+W]**, visualizzata nel trend del punto 1.

Per mostrare un solo run per slot usa `RUN_SELECTION_MODE="first_per_slot"` (o `"last_per_slot"`).


In [ ]:
if runs_df.empty or assignments_df.empty:
    print("Dati insufficienti per i grafici stacked")
else:
    if not selected_run_ids:
        print("Nessun run selezionato con le opzioni correnti")
    for run_id in selected_run_ids:
        fig = plot_solver_execution_stacked(
            run_id=run_id,
            runs_df=runs_df,
            assignments_df=assignments_df,
            slot_metrics_df=slot_metrics_df,
        )
        plt.show()


## 3) Debug eventi di strict infeasibility

Questa sezione serve a capire quando il vincolo strict sulla finestra errore
non consente l'assegnamento completo e costringe al retry rilassato.


In [ ]:
if debug_df.empty:
    print("Nessun evento strict-infeasible loggato")
else:
    fig = plot_infeasibility_overview(debug_df)
    plt.show()


In [ ]:
if debug_df.empty:
    print("Nessun evento strict-infeasible loggato")
else:
    selected_event_id = str(INFEASIBILITY_EVENT_ID) if INFEASIBILITY_EVENT_ID is not None else str(debug_df.iloc[-1]["event_id"])
    display(debug_df[debug_df["event_id"].astype(str) == selected_event_id])
    fig = plot_infeasibility_event(selected_event_id, debug_df)
    plt.show()


## Note

Se vuoi aggiornare i grafici, rilancia prima `python main.py --duration ...` e poi riesegui il notebook.

Per i primi slot, il solver può usare pre-history virtuale: per questo la media errore "modeled" e quella "real" possono differire.
